# Bengali Handwritten Character Recognition with KNN

This notebook identifies 46 Bengali characters—11 vowels (স্বরবর্ণ) and 35 consonants (ব্যঞ্জনবর্ণ)—from handwritten images with a K-Nearest Neighbors (KNN) classifier. The dataset is included in this repository under `data/`; keep the notebook in the project folder and run it from top to bottom.

## 1 — Install packages

In [15]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn Pillow joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2 — Import libraries and set project options

In [16]:
from pathlib import Path
from collections import Counter
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

# The dataset is included in this repository
DATA_DIRECTORY = Path('data')
TRAIN_SPLIT = 'train'
TEST_SPLIT = 'test'

# Map every romanized dataset folder to its correct Bengali letter.
# The first 11 entries are vowels; the remaining 35 are consonants.
CLASS_FOLDER_TO_LABEL = {
    'o': 'অ', 'aa': 'আ', 'i': 'ই', 'ii': 'ঈ', 'u': 'উ', 'uu': 'ঊ',
    'ri': 'ঋ', 'e': 'এ', 'oi': 'ঐ', 'oo': 'ও', 'ou': 'ঔ',
    'ka': 'ক', 'kha': 'খ', 'ga': 'গ', 'gha': 'ঘ', 'nga': 'ঙ',
    'cha': 'চ', 'chha': 'ছ', 'ja': 'জ', 'jha': 'ঝ', 'nya': 'ঞ',
    'tta': 'ট', 'ttha': 'ঠ', 'dda': 'ড', 'rra': 'ড়', 'ddha': 'ঢ',
    'rha': 'ঢ়', 'nna': 'ণ', 'ta': 'ত', 'tha': 'থ', 'da': 'দ',
    'dha': 'ধ', 'na': 'ন', 'pa': 'প', 'pha': 'ফ', 'ba': 'ব',
    'bha': 'ভ', 'ma': 'ম', 'ya': 'য', 'ra': 'র', 'la': 'ল',
    'sha': 'শ', 'ssa': 'ষ', 'sa': 'স', 'ha': 'হ', 'yya': 'য়',
}
CLASS_LABELS = list(CLASS_FOLDER_TO_LABEL.values())
LABEL_TO_FOLDER = {label: folder for folder, label in CLASS_FOLDER_TO_LABEL.items()}
# Use romanized names in plots because the default Matplotlib font may not contain Bengali glyphs.
DISPLAY_NAMES = [LABEL_TO_FOLDER[label] for label in CLASS_LABELS]
print(f'Configured classes: {len(CLASS_LABELS)} (11 vowels and 35 consonants)')

# A balanced sample makes KNN fast.
MAX_TRAIN_PER_CLASS = 180
MAX_TEST_PER_CLASS = 60
IMAGE_SIZE = (32, 32)
RANDOM_STATE = 55
sns.set_theme(style='whitegrid')

## 3 — Find the data folders

In [17]:
def find_split(split_name):
    matches = [p for p in DATA_DIRECTORY.rglob(split_name) if p.is_dir() and any(x.is_dir() for x in p.iterdir())]
    if not matches:
        raise FileNotFoundError(
            f"Cannot find '{split_name}'. Download or clone the complete repository so data/real_data/{split_name}/ is beside this notebook."
        )
    return matches[0]


def make_label_map():
    folder_map = dict(CLASS_FOLDER_TO_LABEL)
    folder_map.update({label: label for label in CLASS_LABELS})

    # A label-map file is needed only when folders use numbers such as 00 or 01.
    for csv_file in DATA_DIRECTORY.rglob('*label*map*.csv'):
        table = pd.read_csv(csv_file)
        letter_columns = [c for c in table.columns if table[c].astype(str).isin(CLASS_LABELS).sum() >= len(CLASS_LABELS)]
        if not letter_columns:
            continue
        letter_column = letter_columns[0]
        id_columns = [c for c in table.columns if c.lower() in {'id', 'label', 'class', 'class_id', 'classid', 'index'}]
        if not id_columns:
            id_columns = [c for c in table.columns if c != letter_column]

        for _, row in table.iterrows():
            letter = str(row[letter_column])
            if letter not in CLASS_LABELS:
                continue
            for id_column in id_columns:
                value = str(row[id_column])
                aliases = {value, f'class_{value}', f'label_{value}'}
                try:
                    number = int(float(value))
                    aliases.update({str(number), f'{number:02d}', f'{number:03d}'})
                except ValueError:
                    pass
                for alias in aliases:
                    folder_map[alias] = letter
        print(f'Using label map: {csv_file}')
        break
    return folder_map


train_folder = find_split(TRAIN_SPLIT)
test_folder = find_split(TEST_SPLIT)
folder_to_label = make_label_map()
print('Training folder:', train_folder)
print('Test folder:', test_folder)

Training folder: data\real_data\train
Test folder: data\real_data\test


## 4 — Select a balanced set of character images

In [18]:
def collect_images(split_folder):
    # Create (image path, Bengali letter) pairs for all 46 target classes.
    pairs = []
    for class_folder in split_folder.iterdir():
        if not class_folder.is_dir():
            continue
        label = folder_to_label.get(class_folder.name, class_folder.name)
        if label in CLASS_LABELS:
            pairs.extend((image_path, label) for image_path in class_folder.glob('*.png'))
    found = Counter(label for _, label in pairs)
    missing = set(CLASS_LABELS) - set(found)
    if missing:
        raise ValueError(f'Missing class folders: {sorted(missing)}. Check data/ and the folder mapping.')
    print('Images available per character:', dict(sorted(found.items())))
    return pairs


def balanced_sample(pairs, maximum_per_class, seed):
    # Choose the same amount of random images from every character class.
    rng = np.random.default_rng(seed)
    chosen = []
    for character in CLASS_LABELS:
        paths = [path for path, label in pairs if label == character]
        count = min(maximum_per_class, len(paths))
        selected = rng.choice(len(paths), count, replace=False)
        chosen.extend((paths[i], character) for i in selected)
    rng.shuffle(chosen)
    return chosen


train_pairs = balanced_sample(collect_images(train_folder), MAX_TRAIN_PER_CLASS, RANDOM_STATE)
test_pairs = balanced_sample(collect_images(test_folder), MAX_TEST_PER_CLASS, RANDOM_STATE + 1)
print('Selected training images:', len(train_pairs))
print('Selected test images:', len(test_pairs))

ValueError: Missing vowel folders: ['অ', 'আ', 'ই', 'ঈ', 'উ', 'ঊ', 'ঋ', 'এ', 'ঐ', 'ও', 'ঔ']. Check data/ and label_map.csv.

## 5 — Load, normalize, and view images

In [ ]:
def load_images(pairs):
    vectors, labels = [], []
    for image_path, label in pairs:
        # L means grayscale; resize keeps every input image the same size.
        with Image.open(image_path) as image:
            pixels = np.asarray(image.convert('L').resize(IMAGE_SIZE), dtype=np.float32)
        vectors.append(pixels.reshape(-1) / 255.0)
        labels.append(label)
    return np.vstack(vectors), np.asarray(labels)


X_train, y_train = load_images(train_pairs)
X_test, y_test = load_images(test_pairs)
print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)

fig, axes = plt.subplots(5, 9, figsize=(16, 9))
for axis, character in zip(axes.flat, CLASS_LABELS):
    index = np.where(y_train == character)[0][0]
    axis.imshow(X_train[index].reshape(IMAGE_SIZE), cmap='gray')
    axis.set_title(LABEL_TO_FOLDER[character], fontsize=9)
    axis.axis('off')
for axis in axes.flat[len(CLASS_LABELS):]:
    axis.axis('off')
plt.tight_layout()
plt.show()

## 6 — Train KNN and choose the best K

In [ ]:
pipeline = Pipeline([
    ('pca', PCA(n_components=0.90, svd_solver='full', random_state=RANDOM_STATE)),
    ('knn', KNeighborsClassifier(n_jobs=-1)),
])

search = GridSearchCV(
    pipeline,
    param_grid={'knn__n_neighbors': [1, 3, 5, 7, 9], 'knn__weights': ['uniform', 'distance']},
    scoring='balanced_accuracy',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1,
)

start = time.perf_counter()
search.fit(X_train, y_train)
best_model = search.best_estimator_
print(f'Training time: {time.perf_counter() - start:.2f} seconds')
print('Best settings:', search.best_params_)
print(f'Validation balanced accuracy: {search.best_score_:.2%}')

results = pd.DataFrame(search.cv_results_)
score_by_k = results.groupby('param_knn__n_neighbors', as_index=False)['mean_test_score'].max()
plt.plot(score_by_k['param_knn__n_neighbors'], score_by_k['mean_test_score'], marker='o')
plt.xlabel('Number of neighbours (K)')
plt.ylabel('Cross-validation balanced accuracy')
plt.title('Choosing K')
plt.ylim(0, 1.05)
plt.show()

## 7 — Evaluate the model on unseen test images

In [ ]:
y_pred = best_model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.2%}')
print(f'Test balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.2%}')
print('\nClassification report:\n')
print(classification_report(y_test, y_pred, labels=CLASS_LABELS, target_names=DISPLAY_NAMES, zero_division=0))

matrix = confusion_matrix(y_test, y_pred, labels=CLASS_LABELS)
plt.figure(figsize=(18, 15))
sns.heatmap(matrix, annot=True, fmt='d', cmap='YlGnBu', xticklabels=DISPLAY_NAMES, yticklabels=DISPLAY_NAMES)
plt.xlabel('Predicted character (romanized)')
plt.ylabel('Actual character (romanized)')
plt.title('Confusion matrix: 46 Bengali characters')
plt.show()

## 8 — Inspect a model mistake and save the trained model

In [ ]:
correct = np.flatnonzero(y_pred == y_test)
query_index = int(correct[0]) if len(correct) else 0
pca = best_model.named_steps['pca']
knn = best_model.named_steps['knn']
distances, neighbour_indices = knn.kneighbors(pca.transform(X_test[[query_index]]))

fig, axes = plt.subplots(1, len(neighbour_indices[0]) + 1, figsize=(12, 3))
axes[0].imshow(X_test[query_index].reshape(IMAGE_SIZE), cmap='gray')
axes[0].set_title(f'Query\ntrue: {LABEL_TO_FOLDER[y_test[query_index]]}\npred: {LABEL_TO_FOLDER[y_pred[query_index]]}')
axes[0].axis('off')
for position, (neighbour, distance) in enumerate(zip(neighbour_indices[0], distances[0]), start=1):
    axes[position].imshow(X_train[neighbour].reshape(IMAGE_SIZE), cmap='gray')
    axes[position].set_title(f'Neighbour {position}\n{LABEL_TO_FOLDER[y_train[neighbour]]}\nd={distance:.2f}')
    axes[position].axis('off')
plt.tight_layout()
plt.show()

mistakes = np.flatnonzero(y_pred != y_test)
if len(mistakes):
    index = mistakes[0]
    plt.imshow(X_test[index].reshape(IMAGE_SIZE), cmap='gray')
    plt.title(f'Actual: {LABEL_TO_FOLDER[y_test[index]]} | Predicted: {LABEL_TO_FOLDER[y_pred[index]]}')
    plt.axis('off')
    plt.show()
else:
    print('No errors in this small test sample. Increase MAX_TEST_PER_CLASS to test more images.')

MODEL_PATH = Path('models/bengali_vowel_knn.joblib')
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(best_model, MODEL_PATH)
print(f'Model saved to: {MODEL_PATH}')